In [ ]:
using Bessels, MPSDynamics, ChaosTools, ProgressBars
include("/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/config/requirements.jl");

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 30.0046747524318
ω_b_0 = 2*ω_a#7.0515*2*pi

Lamb_shift = 0.05 #Lamb shift in GHz


φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.204
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b_0
E_Cb = C_b_inv/2

E_J = 37.12*2*pi*1.5


##Drive


ω_d=ω_b_0#+Lamb_shift*2*pi # Drive frequency
ω_p=2ω_a-ω_d

ϵ_0 = 0.61223
g2 = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10.12*g2

buffer_phase = 0.0
pump_phase = 0.0;

##Env

κ_b=1/10.4; # 1/ns from Leon's paper
κ_2 = 4*g2^2/κ_b
κ_a = 0#3e-3*κ_2 #from Leon's paper

In [ ]:
t_max = 150/κ_b
t_min = 0.0
t_min_save = t_max - 30/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2/(5sqrt(2)*ω_a)
t_save = max(t_min_save,0):dt_save:t_max

re_a_0, im_a_0 = -4, 0
a_0 = re_a_0 +1im*im_a_0
re_b_0, im_b_0 = 0, 0
b_0 = re_b_0 +1im*im_b_0

system = SystemATS_ladder(ω_a, ω_b_0, E_J, E_Ca, E_Cb, a_0, b_0)
system_prepared = prepare(system)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment);

u_0 = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

a, b, t_list = default_solver(u_0, (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)
a_rot = a.*exp.(1im*ω_a*t_list);

In [ ]:
plot(real.(a_rot), imag.(a_rot))

In [ ]:
t_max = 150/κ_b
t_min = 0.0
t_min_save = t_max - 10/κ_b
tspan = (0.0, t_max) # Time span for the simulation
dt = 1 / (50 * ω_a) # Time step for the simulation

dt_save=2pi/(ω_a)
t_save = max(t_min_save,0):dt_save:t_max


re_a_0, im_a_0 = -4, 0
ϕ_a_0, q_a_0=sqrt(2*Z_a)*re_a_0, sqrt(2/Z_a)*im_a_0
re_b_0, im_b_0 = 0, 0
ϕ_b_0, q_b_0=sqrt(2*Z_b)*re_b_0, sqrt(2/Z_b)*im_b_0

system = SystemATS(ω_a, ω_b_0, E_J, E_Ca, E_Cb, ϕ_a_0, q_a_0, ϕ_b_0, q_b_0)
system_prepared = prepare(system)
drive = DriveTwoModesBasics(ϵ_d, ω_d, buffer_phase, ϵ_0, ω_p, pump_phase)
drive_prepared = prepare(drive)
environment = MeanField(κ_a, κ_b)
environment_prepared = prepare(system,environment);

u_dot_0, u_0 = get_init(system_prepared, environment_prepared)

p = SimulationParams(system_prepared, drive_prepared, environment_prepared)

ϕ_a, ϕ_b, q_a, q_b, t_list = default_solver((u_dot_0, u_0), (t_min, t_max), p, dt, t_save = t_save, equation = default_equation!)

In [ ]:
pi*sqrt(2)*5/24

In [ ]:
p1,p2 = plot_mode(ϕ_a, q_a, t_list, ω_a, Z_a)

plot_perso(p1,p2,layout=(1,2),
xlabel=L"\Re(a)", ylabel=L"\Im(a)",
title=["Resonator a" "Resonator a RF"],
legend=false)

In [ ]:
Re = ϕ_a / sqrt(2*Z_a)
Im = q_a * sqrt(Z_a/2)

ω =  ω_a
Re_2 =@. Re * cos(ω * t_list) + Im * sin(-ω * t_list)
Im_2 =@. -Re * sin(-ω * t_list) + Im * cos(ω * t_list);

In [ ]:
Re_2[1:70]

In [ ]:
start = 1
stop = length(Re_2)
step = 100
scatter(Re_2[1:1],Im_2[1:1], ms=5)
scatter!(Re_2[start:step:stop],Im_2[start:step:stop], zcolor = t_list[start:step:stop]*κ_b, ms=1)